# 1.DataSet Processing

In [1]:
import json
import os
import pandas as pd
import numpy as np
import nltk
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [7]:
nltk.download('punkt')
stop_words = set(stopwords.words('english'))
stemmer = PorterStemmer()

def preprocess_text(text, remove_stopwords=True):
	"""Preprocesses text by lowercasing, tokenizing, removing stopwords and stemming."""
	stemmer = PorterStemmer()
	tokens = word_tokenize(text.lower())
	if remove_stopwords:
		stop_words = set(stopwords.words('english'))
		filtered_tokens = [stemmer.stem(word) for word in tokens if word.isalnum() and word not in stop_words]
	else:
		filtered_tokens = [stemmer.stem(word) for word in tokens if word.isalnum()]
	return ' '.join(filtered_tokens)
	
# Load JSON data
def load_data(filepath):
	with open(filepath, 'r') as file:
		data = json.load(file)
	return data

[nltk_data] Downloading package punkt to /Users/chenluyao/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [5]:
def convert_to_df(data, labelled=True, remove_stopwords=True):
	data_for_dataframe = []
	for claim_id, claim_details in data.items():
		claim_text = preprocess_text(claim_details['claim_text'], remove_stopwords)
		claim_label = claim_details['claim_label']
		eids = claim_details['evidences']
		if labelled:
			data_for_dataframe.append({
					'claim_id': claim_id,
					'claim_text': claim_details['claim_text'],
					'claim_preprocessed': claim_text,
					'evidence': eids,
					'label': claim_label
				})
		else:
			data_for_dataframe.append({
					'claim_id': claim_id,
					'claim_text': claim_details['claim_text'],
					'claim_preprocessed': claim_text,
				})

	# create DataFrame
	df = pd.DataFrame(data_for_dataframe)
	return df

In [12]:
train_claims_data = load_data('../data/train-claims.json')
evidence_data = load_data('../data/evidence.json')
dev_claims_data = load_data('../data/dev-claims.json')

train_claims_df = convert_to_df(train_claims_data, labelled=True, remove_stopwords=True)
dev_claims_df = convert_to_df(dev_claims_data, labelled=False, remove_stopwords=True)

evidence_map = load_data('../data/curated/preprocessed_evidence_map.json')
evidence_df = pd.DataFrame(evidence_map.items(), columns=['id', 'evidence'])

train_claims_text = train_claims_df['claim_preprocessed'].tolist()

dev_claims_text = dev_claims_df['claim_preprocessed'].tolist()
dev_claims_id = dev_claims_df['claim_id'].tolist()

evidence_id = list(evidence_map.keys())
evidence_text  = list(evidence_map.values())

vectorizer = TfidfVectorizer()
vectorizer.fit(train_claims_text + evidence_text)
evidence_vec = vectorizer.transform(evidence_text)
dev_claims_vec = vectorizer.transform(dev_claims_text)
print(dev_claims_vec.shape)
print(evidence_vec.shape)

(154, 506699)
(1208827, 506699)


In [10]:
sim = cosine_similarity(dev_claims_vec, evidence_vec)

In [11]:
data = np.zeros((sim.shape[0], 3))
top_evidence_id = {}
for i in range(sim.shape[0]):
	data[i] = np.argpartition(sim[i], -3)[-3:]
	top_evidence_id[dev_claims_id[i]] = [evidence_df.iloc[int(ind)]['id'] for ind in data[i]]


## Do not run the cells below

In [30]:
def top_k_evidence(claim_df, evidence_df, evidence_map, k=5):
	# compute cosine similarity between each claim and each evidence
	X = np.array(claim_df['claim_tfidf'].values.tolist())
	y = np.array(evidence_df['evidence_tfidf'].values.tolist())
	sim = cosine_similarity(X, y)

	# get top k evidence with highest similarity score with the claim
	data = np.zeros((sim.shape[0], k))
	top_evidence_id = []
	for i in range(sim.shape[0]):
		data[i] = np.argpartition(sim[i], -k)[-k:]
		top_evidence_id.append([evidence_df.iloc[int(ind)]['id'] for ind in data[i]])

	claim_df['top5_evidence_id'] = top_evidence_id

	claim_df = claim_df[["claim_id", "claim_text_raw", "top5_evidence_id"]]

	# get texts of top k evidence
	claim_df['evidence_texts'] = claim_df['top5_evidence_id'].apply(
		lambda x: [evidence_map[evidence_id] for evidence_id in x]
	)
	return claim_df


In [ ]:
dev_claims_df = top_k_evidence(dev_claims_df, filtered_evidence_df, filtered_evidence_map)
dev_claims_df.to_csv("data/curated/dev_evidence_retrieval.csv", index=False)

In [31]:
test_claims_df = top_k_evidence(test_claims_df, filtered_evidence_df, filtered_evidence_map)
test_claims_df.to_csv("data/curated/test_evidence_retrieval2.csv", index=False)
test_claims_df

/var/folders/df/4qk5nt6555bggnc39502n5b80000gn/T/ipykernel_37917/2447078132.py:19: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  claim_df['evidence_texts'] = claim_df['top5_evidence_id'].apply(


,claim_id,claim_text_raw,top5_evidence_id,evidence_texts
0,claim-2967,The contribution of waste heat to the global c...,"[evidence-1090341, evidence-621274, evidence-1...",[global climat forum a platform for joint stud...
1,claim-979,“Warm weather worsened the most recent five-ye...,"[evidence-417353, evidence-1119884, evidence-8...",[climat is the statist usual mean or variabl o...
2,claim-1609,Greenland has only lost a tiny fraction of its...,"[evidence-891137, evidence-487116, evidence-11...",[the gradual accumul of ice on the laurentid i...
3,claim-1020,“The global reef crisis does not necessarily m...,"[evidence-151390, evidence-1152843, evidence-7...","[myllyoja liter mean millditch, mean absolut p..."
4,claim-2599,Small amounts of very active substances can ca...,"[evidence-347797, evidence-1004410, evidence-7...",[the wolff chaikoff effect is an effect mean o...
...,...,...,...,...
148,claim-293,When the measuring equipment gets old and need...,"[evidence-751374, evidence-104953, evidence-48...",[the new measur requir climat model paramet ad...
149,claim-910,"The cement, iron and steel, and petroleum refi...","[evidence-114183, evidence-42541, evidence-379...",[the theme of hi work is to live a happi prosp...
150,claim-2815,A new peer-reviewed study on Surface Warming a...,"[evidence-500457, evidence-694951, evidence-18...",[the solar storm of known as the carrington ev...
151,claim-1652,The strong CO2 effect has been observed by man...,"[evidence-429904, evidence-16131, evidence-229...",[clinomet measur both inclin posit slope as se...


### Claim Classification

In [42]:
data_for_dataframe = []
for claim_id, claim_details in dev_claims_data.items():
    claim_text = claim_details['claim_text']
    claim_label = claim_details['claim_label']
    eids = claim_details['evidences']
    data_for_dataframe.append({
            'claim': claim_text,
            'evidence': eids,
            'label': claim_label
        })

# Create DataFrame
dev_claims_df = pd.DataFrame(data_for_dataframe)

dev_claims_df['evidence_texts'] = dev_claims_df['evidence'].apply(
    lambda x: [evidence_map[evidence_id] for evidence_id in x]
)

dev_claims_df

,claim,evidence,label,evidence_texts
0,[South Australia] has the most expensive elect...,"[evidence-67732, evidence-572512]",SUPPORTS,[citat need south australia highest retail pri...
1,when 3 per cent of total annual global emissio...,"[evidence-996421, evidence-1080858, evidence-2...",NOT_ENOUGH_INFO,[unep green economi report state agricultur op...
2,This means that the world is now 1C warmer tha...,"[evidence-889933, evidence-694262]",SUPPORTS,[multipl independ produc instrument dataset co...
3,"“As it happens, Zika may also be a good model ...","[evidence-422399, evidence-702226, evidence-28...",NOT_ENOUGH_INFO,[genet disord result deleteri mutat due sponta...
4,Greenland has only lost a tiny fraction of its...,"[evidence-52981, evidence-264761, evidence-947...",REFUTES,[iceberg calv happen averag greenland lost gt ...
...,...,...,...,...
149,"'To suddenly label CO2 as a ""pollutant"" is a d...","[evidence-409365, evidence-127519, evidence-85...",REFUTES,[state articl convent requir greenhou ga ghg c...
150,"after a natural orbitally driven warming, atmo...","[evidence-368192, evidence-261690, evidence-20...",NOT_ENOUGH_INFO,[increa atmosph concentr co greenhou gase meth...
151,Many of the world’s coral reefs are already ba...,"[evidence-1124018, evidence-995813, evidence-1...",NOT_ENOUGH_INFO,[tropic water contain nutrient yet coral reef ...
152,A recent study led by Lawrence Livermore Natio...,[evidence-660755],REFUTES,[studi david douglass cowork conclud commonli ...


In [43]:
# combine claim text and evidence texts
X_train = train_claims_df['claim'] + train_claims_df['evidence_texts'].apply(lambda x: ' '.join(x))
y_train = train_claims_df['label']

X_dev = dev_claims_df['claim'] + dev_claims_df['evidence_texts'].apply(lambda x: ' '.join(x))
y_dev = dev_claims_df['label']

X_test = test_claims_df['claim_text_raw'] + test_claims_df['evidence_texts'].apply(lambda x: ' '.join(x))

count_vectorizer = CountVectorizer()
X_train_count = count_vectorizer.fit_transform(X_train)
X_dev_count = count_vectorizer.transform(X_dev)
X_test_count = count_vectorizer.transform(X_test)

In [46]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# Hyperparameters
n_estimators_values = [50, 100, 200]
max_depth_values = [None, 10, 20]

accuracy_scores_rf = []
for n_estimators in n_estimators_values:
    for max_depth in max_depth_values:
        rf_classifier = RandomForestClassifier(n_estimators=n_estimators, max_depth=max_depth, random_state=42)
        rf_classifier.fit(X_train_count, y_train)
        y_pred_rf = rf_classifier.predict(X_dev_count)
        
        accuracy_rf = accuracy_score(y_dev, y_pred_rf)
        accuracy_scores_rf.append(((n_estimators, max_depth), accuracy_rf))
        print(f"n_estimators = {n_estimators}, max_depth = {max_depth}: Accuracy = {accuracy_rf}")

print("Accuracy scores for Random Forest:")
for params, accuracy in accuracy_scores_rf:
    print(f"Parameters: {params}, Accuracy: {accuracy}")

n_estimators = 50, max_depth = None: Accuracy = 0.43506493506493504
n_estimators = 50, max_depth = 10: Accuracy = 0.44155844155844154
n_estimators = 50, max_depth = 20: Accuracy = 0.44805194805194803
n_estimators = 100, max_depth = None: Accuracy = 0.42857142857142855
n_estimators = 100, max_depth = 10: Accuracy = 0.44805194805194803
n_estimators = 100, max_depth = 20: Accuracy = 0.44155844155844154
n_estimators = 200, max_depth = None: Accuracy = 0.4155844155844156
n_estimators = 200, max_depth = 10: Accuracy = 0.44155844155844154
n_estimators = 200, max_depth = 20: Accuracy = 0.44155844155844154
Accuracy scores for Random Forest:
Parameters: (50, None), Accuracy: 0.43506493506493504
Parameters: (50, 10), Accuracy: 0.44155844155844154
Parameters: (50, 20), Accuracy: 0.44805194805194803
Parameters: (100, None), Accuracy: 0.42857142857142855
Parameters: (100, 10), Accuracy: 0.44805194805194803
Parameters: (100, 20), Accuracy: 0.44155844155844154
Parameters: (200, None), Accuracy: 0.4155

In [ ]:
# Random Forest 
rf_classifier = RandomForestClassifier(n_estimators=50, max_depth=20, random_state=42)
rf_classifier.fit(X_train_count, y_train)
y_pred = rf_classifier.predict(X_test_count)
test_claims_df["label"] = y_pred
test_claims_df['evidences'] = test_claims_df['top5_evidence_id'].apply(
    lambda x: ["evidence-" + str(evidence_id) for evidence_id in x]
)
test_claims_df.drop(columns=['evidence_texts', 'top5_evidence_id'], inplace=True)
test_claims_df.rename(columns={"claim_text_raw": "claim_text", "label": "claim_label"}, inplace=True)
test_claims_df.set_index('claim_id', inplace=True)
test_claims_df

In [48]:
# convert to json file
from json import loads
result = test_claims_df.to_json(orient="index")
with open('data/curated/test-output2.json', 'w') as f:
    f.write(result)